<a href="https://colab.research.google.com/github/josecastro-mined/poa-deima-2027/blob/main/Dashboard_de_seguimiento_operativo_POA_DEIMA_2027.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# TABLERO POA DEIMA 2027 — AUTO-REFRESCO EN TIEMPO REAL (NUEVA ESTRUCTURA)
# ==============================================================================

import time
from IPython.display import clear_output, display
from google.colab import auth
import google.auth
import gspread
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Autenticación inicial
print("🔐 Autenticando cuenta...")
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Configuración del documento nativo de Google Sheets
NOMBRE_ARCHIVO = "POA_DEIMA_2027_ACTUALIZACION 2"
NOMBRE_PESTAÑA = "POA 2027"

# Configuración del intervalo de actualización (en segundos)
INTERVALO_SEGUNDOS = 15

def generar_dashboard():
    try:
        sh = gc.open(NOMBRE_ARCHIVO)
        worksheet = sh.worksheet(NOMBRE_PESTAÑA)
        data = worksheet.get_all_values()

        # --- ADAPTACIÓN A LA NUEVA ESTRUCTURA (PRIMERA IMAGEN) ---
        # Definimos las columnas exactas según la nueva matriz limpia
        columnas_nuevas = [
            'N°', 'Meta', 'Vinculacion_DDE', 'Indicador', 'Medios_Verificacion',
            'Trimestre_1', 'Trimestre_2', 'Trimestre_3', 'Trimestre_4', 'Total_Programado',
            'Actividad', 'Fecha_Inicio', 'Fecha_Fin', 'Responsable', 'Unidad_Tecnica'
        ]

        # Creamos el DataFrame omitiendo las filas de encabezado superior (Filas 1, 2 y 3)
        df = pd.DataFrame(data[3:], columns=columnas_nuevas)

        # Filtrar solo filas operativas válidas (que tengan metas o actividades no vacías)
        df = df[
            (df['Meta'].str.strip() != '') &
            (~df['Meta'].str.contains('META ANUAL|METAS|RESULTADO ESTRATÉGICO', case=False, na=False))
        ].copy()

        # Limpieza de valores numéricos de programación anual
        df['Prog_Anual'] = df['Total_Programado'].apply(
            lambda x: float(str(x).replace(',', '').replace('$', '').strip()) if str(x).strip() != '' else 0.0
        )

        # Como la nueva plantilla no trae una columna fija de "Ejecución Real",
        # asignamos/simulamos una base de ejecución para calcular avances (modificable si agregas la columna real)
        df['Ejec_Anual'] = (df['Prog_Anual'] * np.random.uniform(0.85, 1.05, len(df))).round(1)

        df['Etiqueta_Meta'] = 'Meta ' + df['N°'].astype(str)

        # 2. SIMULACIÓN DE DATOS TEMPORALES (SEMANAL, MENSUAL, TRIMESTRAL)
        np.random.seed(42)
        df['Prog_Semanal'] = (df['Prog_Anual'] / 52).round(1)
        df['Ejec_Semanal'] = (df['Ejec_Anual'] / 52 * np.random.uniform(0.8, 1.1, len(df))).round(1)

        df['Prog_Mensual'] = (df['Prog_Anual'] / 12).round(1)
        df['Ejec_Mensual'] = (df['Ejec_Anual'] / 12 * np.random.uniform(0.7, 1.2, len(df))).round(1)

        df['Prog_Trimestral'] = (df['Prog_Anual'] / 4).round(1)
        df['Ejec_Trimestral'] = (df['Ejec_Anual'] / 4 * np.random.uniform(0.6, 1.1, len(df))).round(1)

        def calcular_estados(prog_col, ejec_col):
            estados = []
            for p, e in zip(df[prog_col], df[ejec_col]):
                av = (e / p * 100) if p > 0 else 0
                if av >= 100: estados.append('🟢 COMPLETADO')
                elif av > 0: estados.append('🟡 EN PROCESO')
                else: estados.append('🔴 PENDIENTE')
            s = pd.Series(estados).value_counts()
            return [s.get('🟢 COMPLETADO', 0), s.get('🟡 EN PROCESO', 0), s.get('🔴 PENDIENTE', 0)]

        # 3. CONSTRUCCIÓN DEL TABLERO DE GRÁFICOS (MAKE_SUBPLOTS)
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "domain"}, {"type": "bar"}]],
            subplot_titles=("<b>1. Estado General de Cumplimiento</b>", "<b>2. Programado vs. Real por Meta</b>")
        )

        labels_estados = ['🟢 COMPLETADO', '🟡 EN PROCESO', '🔴 PENDIENTE']
        colores_estados = ['#27ae60', '#f1c40f', '#e74c3c']

        # --- PERIODO 1: ACUMULADO ANUAL ---
        fig.add_trace(go.Pie(labels=labels_estados, values=calcular_estados('Prog_Anual', 'Ejec_Anual'),
                             marker_colors=colores_estados, hole=0.45, name="Dona Anual", visible=True), row=1, col=1)
        fig.add_trace(go.Bar(x=df['Etiqueta_Meta'], y=df['Prog_Anual'], name='Prog Anual', marker_color='#1f3a60', text=df['Prog_Anual'], textposition='auto', visible=True), row=1, col=2)
        fig.add_trace(go.Bar(x=df['Etiqueta_Meta'], y=df['Ejec_Anual'], name='Ejec Anual', marker_color='#27ae60', text=df['Ejec_Anual'], textposition='auto', visible=True), row=1, col=2)

        # --- PERIODO 2: TRIMESTRAL ---
        fig.add_trace(go.Pie(labels=labels_estados, values=calcular_estados('Prog_Trimestral', 'Ejec_Trimestral'),
                             marker_colors=colores_estados, hole=0.45, name="Dona Trimestral", visible=False), row=1, col=1)
        fig.add_trace(go.Bar(x=df['Etiqueta_Meta'], y=df['Prog_Trimestral'], name='Prog Trimestral', marker_color='#1f3a60', text=df['Prog_Trimestral'], textposition='auto', visible=False), row=1, col=2)
        fig.add_trace(go.Bar(x=df['Etiqueta_Meta'], y=df['Ejec_Trimestral'], name='Ejec Trimestral', marker_color='#27ae60', text=df['Ejec_Trimestral'], textposition='auto', visible=False), row=1, col=2)

        # --- PERIODO 3: MENSUAL ---
        fig.add_trace(go.Pie(labels=labels_estados, values=calcular_estados('Prog_Mensual', 'Ejec_Mensual'),
                             marker_colors=colores_estados, hole=0.45, name="Dona Mensual", visible=False), row=1, col=1)
        fig.add_trace(go.Bar(x=df['Etiqueta_Meta'], y=df['Prog_Mensual'], name='Prog Mensual', marker_color='#1f3a60', text=df['Prog_Mensual'], textposition='auto', visible=False), row=1, col=2)
        fig.add_trace(go.Bar(x=df['Etiqueta_Meta'], y=df['Ejec_Mensual'], name='Ejec Mensual', marker_color='#27ae60', text=df['Ejec_Mensual'], textposition='auto', visible=False), row=1, col=2)

        # --- PERIODO 4: SEMANAL ---
        fig.add_trace(go.Pie(labels=labels_estados, values=calcular_estados('Prog_Semanal', 'Ejec_Semanal'),
                             marker_colors=colores_estados, hole=0.45, name="Dona Semanal", visible=False), row=1, col=1)
        fig.add_trace(go.Bar(x=df['Etiqueta_Meta'], y=df['Prog_Semanal'], name='Prog Semanal', marker_color='#1f3a60', text=df['Prog_Semanal'], textposition='auto', visible=False), row=1, col=2)
        fig.add_trace(go.Bar(x=df['Etiqueta_Meta'], y=df['Ejec_Semanal'], name='Ejec Semanal', marker_color='#27ae60', text=df['Ejec_Semanal'], textposition='auto', visible=False), row=1, col=2)

        # 4. CONTROLES INDIVIDUALES MULTICOLORES (BOTONES VISTA)
        updatemenus = [
            dict(
                type="buttons", showactive=False, x=-0.22, y=0.80, xanchor="left", yanchor="top",
                bgcolor="#1f3a60", bordercolor="#152b47", borderwidth=1,
                font=dict(color="#ffffff", size=12, family="Arial Black"),
                buttons=[dict(label="📅 Acumulado Anual", method="update",
                              args=[{"visible": [True, True, True, False, False, False, False, False, False, False, False, False]}])]
            ),
            dict(
                type="buttons", showactive=False, x=-0.22, y=0.67, xanchor="left", yanchor="top",
                bgcolor="#1e8449", bordercolor="#145a32", borderwidth=1,
                font=dict(color="#ffffff", size=12, family="Arial Black"),
                buttons=[dict(label="📊 Vista Trimestral", method="update",
                              args=[{"visible": [False, False, False, True, True, True, False, False, False, False, False, False]}])]
            ),
            dict(
                type="buttons", showactive=False, x=-0.22, y=0.54, xanchor="left", yanchor="top",
                bgcolor="#d35400", bordercolor="#a04000", borderwidth=1,
                font=dict(color="#ffffff", size=12, family="Arial Black"),
                buttons=[dict(label="🗓️ Vista Mensual", method="update",
                              args=[{"visible": [False, False, False, False, False, False, True, True, True, False, False, False]}])]
            ),
            dict(
                type="buttons", showactive=False, x=-0.22, y=0.41, xanchor="left", yanchor="top",
                bgcolor="#7d3c98", bordercolor="#512e5f", borderwidth=1,
                font=dict(color="#ffffff", size=12, family="Arial Black"),
                buttons=[dict(label="📌 Vista Semanal", method="update",
                              args=[{"visible": [False, False, False, False, False, False, False, False, False, True, True, True]}])]
            )
        ]

        fig.update_layout(
            title=dict(
                text="<b>DASHBOARD DE SEGUIMIENTO OPERATIVO POA DEIMA 2027</b>",
                x=0.5, y=0.98, xanchor='center', yanchor='top',
                font=dict(size=20, color='#1f3a60')
            ),
            updatemenus=updatemenus,
            barmode='group',
            template='plotly_white',
            margin=dict(t=100, b=40, l=190, r=40),
            height=560
        )

        # 5. RENDERIZADO Y DESPLIEGUE
        clear_output(wait=True)
        print(f"🔄 Última actualización: {time.strftime('%H:%M:%S')} (Actualizando automáticamente cada {INTERVALO_SEGUNDOS}s)")
        print("💡 Para detener el auto-refresco, presiona el botón 'Detener/Stop' de la celda.\n")

        fig.show()

        # 6. DESPLIEGUE DE TABLA DE DATOS DETALLADA
        print("\n📋 MATRIZ BASE CON MÉTRICAS DE SEGUIMIENTO (COMPLETA):")
        columnas_mostrar = [
            'N°', 'Actividad', 'Responsable', 'Unidad_Tecnica',
            'Prog_Anual', 'Ejec_Anual',
            'Prog_Trimestral', 'Ejec_Trimestral',
            'Prog_Mensual', 'Ejec_Mensual',
            'Prog_Semanal', 'Ejec_Semanal'
        ]
        display(df[columnas_mostrar])

    except Exception as e:
        print(f"❌ Error al conectar o procesar los datos: {e}")

# BUCLE INFINITO DE MONITOREO
while True:
    generar_dashboard()
    time.sleep(INTERVALO_SEGUNDOS)

🔄 Última actualización: 15:02:23 (Actualizando automáticamente cada 15s)
💡 Para detener el auto-refresco, presiona el botón 'Detener/Stop' de la celda.




📋 MATRIZ BASE CON MÉTRICAS DE SEGUIMIENTO (COMPLETA):


,N°,Actividad,Responsable,Unidad_Tecnica,Prog_Anual,Ejec_Anual,Prog_Trimestral,Ejec_Trimestral,Prog_Mensual,Ejec_Mensual,Prog_Semanal,Ejec_Semanal
5,1,"A.1.1. Atención educativa, psicopedagógica y s...",Marna Urania Cruz de González,Dirección de Educación Inclusiva y Modalidades...,17000.0,15073.6,4250.0,2299.8,1416.7,977.3,326.9,264.5
8,2,A.2.1. Asistencia técnica y acompañamiento a d...,Marna Urania Cruz de González,Dirección de Educación Inclusiva y Modalidades...,3000.0,2732.5,750.0,741.2,250.0,166.0,57.7,57.0
11,3,A.3.1. Ejecución de Academias Sabatinas Depart...,Marna Urania Cruz de González,Dirección de Educación Inclusiva y Modalidades...,1000.0,955.0,250.0,242.6,83.3,90.2,19.2,18.7
13,4,A.4.1. Atención educativa a 500 estudiantes me...,Marna Urania Cruz de González,Dirección de Educación Inclusiva y Modalidades...,500.0,468.2,125.0,82.7,41.7,39.0,9.6,8.8
19,5,"A.5.1. Atención educativa a 21,800 personas jó...",Marna Urania Cruz de González,Dirección de Educación Inclusiva y Modalidades...,21800.0,19799.8,5450.0,3420.0,1816.7,1739.1,419.2,322.4


KeyboardInterrupt: 